In [ ]:
!pip install deepspeed transformers --quiet
!pip install bitsandbytes accelerate --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 14.3 MB/s eta 0:00:00


In [ ]:
# ----------------------------
# 1. Load CSV
# ----------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# ----------------------------
# 2. LoRA Config
# ----------------------------
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
# ----------------------------
# 3. Load CSV Dataset
# ----------------------------
from datasets import load_dataset

dataset = load_dataset("csv", data_files="medDataset_processed.csv", split="train")



Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# ----------------------------
# 4. Preprocessing
# ----------------------------
def format_example(example):
    prompts = [f"Question: {q}\nAnswer:" for q in example['Question']]
    eos = tokenizer.eos_token if tokenizer.eos_token else "<|endoftext|>"

    full_inputs = [p + " " + a + " " + eos for p, a in zip(prompts, example['Answer'])]

    tokenized = tokenizer(
        full_inputs,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )

    labels = tokenized["input_ids"].clone()
    for i, prompt in enumerate(prompts):
        prompt_len = len(tokenizer(prompt)["input_ids"])
        labels[i, :prompt_len] = -100  # Mask prompt tokens

    tokenized["labels"] = labels
    return tokenized


tokenized_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
    batched=True,
    batch_size=16
)


Map:   0%|          | 0/16407 [00:00<?, ? examples/s]

In [ ]:
# ----------------------------
# 5. Train/Validation Split
# ----------------------------
tokenized_dataset.set_format(type="torch")
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']



In [ ]:
# ----------------------------
# 6. Training Arguments
# ----------------------------
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-medquad-finetuned",
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=42,
    deepspeed="ds_config.json"
)


In [ ]:
# ----------------------------
# 7. Data Collator
# ----------------------------
from transformers import default_data_collator
data_collator = default_data_collator

In [ ]:
# ----------------------------
# 8. Trainer
# ----------------------------
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)



/tmp/ipython-input-1829897253.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
!pip install mpi4py --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.4 MB/s eta 0:00:00


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
W1012 13:04:43.136000 452 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1012 13:04:43.136000 452 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


Installed CUDA version 12.5 does not match the version torch was compiled with 12.6 but since the APIs are compatible, accepting this combination


Step,Training Loss,Validation Loss
2000,1.447500,1.429047


In [ ]:
trainer.save_model("./my-qwen-v4")
tokenizer.save_pretrained("./my-qwen-v4")

In [ ]:
!zip -r model.zip my-qwen-v4/
from google.colab import files
files.download('model.zip')